In [19]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
# options.add_argument("--headless")  # 필요하면 주석 해제

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://www.megabox.co.kr/theater/list")

    wait = WebDriverWait(driver, 10)

    # 강남 지역 선택
    region_buttons = wait.until(
        EC.presence_of_all_elements_located((
            By.CSS_SELECTOR,
            "#contents div.theater-box div.theater-place ul li.on div ul li > a"
        ))
    )
    for btn in region_buttons:
        if "강남" in btn.text:
            btn.click()
            print("강남 지역 선택 완료")
            break

    time.sleep(2)  # 지역 선택 후 페이지 로딩 대기

    # 상세 주소 가져오기 (두 번째 ul.dot-list 사용)
    dot_lists = driver.find_elements(By.CSS_SELECTOR, "#tab01 ul.dot-list")
    address = ""
    if len(dot_lists) > 1:
        li_list = dot_lists[1].find_elements(By.TAG_NAME, "li")
        for li in li_list:
            text = li.text.strip()
            if text.startswith("도로명주소"):
                parts = text.split(":", 1)
                if len(parts) > 1:
                    address = parts[1].strip()
                break

    cinema_name = "강남 메가박스"
    print(f"영화관명: {cinema_name}")
    print(f"상세주소: {address}")

    # 상영시간 탭 클릭 (#tab02 활성화)
    schedule_tab = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "#contents > div.inner-wrap.pt40 > div.tab-list.fixed.mb40.tab-layer > ul > li:nth-child(2) > a"))
    )
    schedule_tab.click()
    time.sleep(2)  # 탭 전환 후 로딩 대기

    # 영화별 상영시간 크롤링
    movie_blocks = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#tab02 div.reserve.theater-list-box > div"))
    )

    for block in movie_blocks:
        try:
            title_elem = block.find_element(By.CSS_SELECTOR, "div.theater-tit p:nth-child(2) a")
            title = title_elem.text.strip()
        except:
            continue

        try:
            time_elems = block.find_elements(By.CSS_SELECTOR, "div.theater-time-box p.time")
            times = [t.text.strip() for t in time_elems if t.text.strip()]
        except:
            times = []

        print(f"🎬 {title}")
        print("🕒 상영시간:", ", ".join(times) if times else "정보 없음")
        print("-" * 40)

finally:
    driver.quit()

강남 지역 선택 완료
영화관명: 강남 메가박스
상세주소: 서울특별시 서초구 서초대로 77길 3 (서초동) 아라타워 8층
🎬 판타스틱 4: 새로운 출발
🕒 상영시간: 12:55, 15:20, 17:45, 20:10, 22:35, 18:50, 21:15
----------------------------------------
🎬 전지적 독자 시점
🕒 상영시간: 12:25, 14:50, 17:15, 19:40, 22:05, 18:30, 20:55
----------------------------------------
🎬 F1 더 무비
🕒 상영시간: 12:15, 15:20, 18:25, 21:30
----------------------------------------
🎬 킹 오브 킹스
🕒 상영시간: 12:10, 17:00
----------------------------------------
🎬 (더빙) 명탐정 코난: 척안의 잔상
🕒 상영시간: 11:00, 15:40
----------------------------------------
🎬 명탐정 코난: 척안의 잔상
🕒 상영시간: 13:20, 18:00, 20:20, 22:40
----------------------------------------
🎬 커미션
🕒 상영시간: 12:05
----------------------------------------
🎬 노이즈
🕒 상영시간: 12:40, 14:20, 16:25
----------------------------------------
🎬 [응원상영] 킹 오브 프리즘 -유어 엔드리스 콜- 모두 빛나라! 프리즘 투어즈
🕒 상영시간: 17:20
----------------------------------------
🎬 쥬라기 월드: 새로운 시작
🕒 상영시간: 14:40, 19:10
----------------------------------------
🎬 슈퍼맨
🕒 상영시간: 14:25
---------------------------------------

In [20]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    driver.get("https://www.megabox.co.kr/theater/list")

    # 모든 지역 버튼들 선택 (강남 포함)
    region_buttons = wait.until(
        EC.presence_of_all_elements_located((
            By.CSS_SELECTOR,
            "#contents div.theater-box div.theater-place ul li.on div ul li > a"
        ))
    )

    all_cinemas = []

    for i, btn in enumerate(region_buttons):
        region_name = btn.text.strip()
        print(f"==> [{i+1}/{len(region_buttons)}] 지역 선택: {region_name}")

        # 지역 클릭
        btn.click()
        time.sleep(2)  # 로딩 대기

        # 상세 주소 추출
        dot_lists = driver.find_elements(By.CSS_SELECTOR, "#tab01 ul.dot-list")
        address = ""
        if len(dot_lists) > 1:
            li_list = dot_lists[1].find_elements(By.TAG_NAME, "li")
            for li in li_list:
                text = li.text.strip()
                if text.startswith("도로명주소"):
                    parts = text.split(":", 1)
                    if len(parts) > 1:
                        address = parts[1].strip()
                    break

        cinema_name = f"{region_name} 메가박스"

        # 상영시간 탭 클릭
        schedule_tab = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "#contents > div.inner-wrap.pt40 > div.tab-list.fixed.mb40.tab-layer > ul > li:nth-child(2) > a"))
        )
        schedule_tab.click()
        time.sleep(2)

        movie_blocks = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#tab02 div.reserve.theater-list-box > div"))
        )

        movies = []
        for block in movie_blocks:
            try:
                title_elem = block.find_element(By.CSS_SELECTOR, "div.theater-tit p:nth-child(2) a")
                title = title_elem.text.strip()
            except:
                continue

            try:
                time_elems = block.find_elements(By.CSS_SELECTOR, "div.theater-time-box p.time")
                times = [t.text.strip() for t in time_elems if t.text.strip()]
            except:
                times = []

            movies.append({"title": title, "times": times})

        cinema_info = {
            "cinema_name": cinema_name,
            "address": address,
            "movies": movies,
        }

        all_cinemas.append(cinema_info)

        print(f"{cinema_name} 정보 수집 완료, 영화 개수: {len(movies)}")
        print("-" * 60)

    # 최종 결과 출력 (간단히)
    for cinema in all_cinemas:
        print(f"■ {cinema['cinema_name']} ({cinema['address']})")
        for movie in cinema["movies"]:
            print(f"  - {movie['title']}: {', '.join(movie['times'])}")
        print()

finally:
    driver.quit()


==> [1/20] 지역 선택: 강남
강남 메가박스 정보 수집 완료, 영화 개수: 12
------------------------------------------------------------


StaleElementReferenceException: Message: stale element reference: stale element not found
  (Session info: chrome=138.0.7204.158); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
	GetHandleVerifier [0x0x7ff60c92e935+77845]
	GetHandleVerifier [0x0x7ff60c92e990+77936]
	(No symbol) [0x0x7ff60c6e9cda]
	(No symbol) [0x0x7ff60c7000f4]
	(No symbol) [0x0x7ff60c6febc3]
	(No symbol) [0x0x7ff60c6f23d9]
	(No symbol) [0x0x7ff60c6f028f]
	(No symbol) [0x0x7ff60c6f471c]
	(No symbol) [0x0x7ff60c6f47ef]
	(No symbol) [0x0x7ff60c73a1b6]
	(No symbol) [0x0x7ff60c7688ca]
	(No symbol) [0x0x7ff60c732f76]
	(No symbol) [0x0x7ff60c768ae0]
	(No symbol) [0x0x7ff60c790b07]
	(No symbol) [0x0x7ff60c7686a3]
	(No symbol) [0x0x7ff60c731791]
	(No symbol) [0x0x7ff60c732523]
	GetHandleVerifier [0x0x7ff60cc0684d+3059501]
	GetHandleVerifier [0x0x7ff60cc00c0d+3035885]
	GetHandleVerifier [0x0x7ff60cc20400+3164896]
	

In [21]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

# 크롬 드라이버 옵션 (필요하면 headless로도 가능)
options = Options()
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

try:
    url = "https://www.megabox.co.kr/theater/list"
    driver.get(url)

    time.sleep(2)  # 페이지 로딩 대기

    # 서울 지역 탭이 기본 활성화 되어있으니 별도 클릭 필요 없음
    # 서울 지역 극장 리스트의 a 태그들을 선택
    elements = driver.find_elements(By.CSS_SELECTOR, "#contents > div > div.theater-box > div.theater-place > ul > li.on > div > ul > li > a")

    href_list = [elem.get_attribute("href") for elem in elements]

    for href in href_list:
        print(href)

finally:
    driver.quit()

https://www.megabox.co.kr/theater?brchNo=1372
https://www.megabox.co.kr/theater?brchNo=1341
https://www.megabox.co.kr/theater?brchNo=0090
https://www.megabox.co.kr/theater?brchNo=1431
https://www.megabox.co.kr/theater?brchNo=0041
https://www.megabox.co.kr/theater?brchNo=0073
https://www.megabox.co.kr/theater?brchNo=1572
https://www.megabox.co.kr/theater?brchNo=1581
https://www.megabox.co.kr/theater?brchNo=1311
https://www.megabox.co.kr/theater?brchNo=1211
https://www.megabox.co.kr/theater?brchNo=1331
https://www.megabox.co.kr/theater?brchNo=1371
https://www.megabox.co.kr/theater?brchNo=1381
https://www.megabox.co.kr/theater?brchNo=1202
https://www.megabox.co.kr/theater?brchNo=1561
https://www.megabox.co.kr/theater?brchNo=1321
https://www.megabox.co.kr/theater?brchNo=1351
https://www.megabox.co.kr/theater?brchNo=1212
https://www.megabox.co.kr/theater?brchNo=0083
https://www.megabox.co.kr/theater?brchNo=1562


In [28]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

def click_region_tab(region_name):
    tabs = driver.find_elements(By.CSS_SELECTOR, "button.sel-city")
    for tab in tabs:
        if tab.text.strip() == region_name:
            tab.click()
            return True
    return False

try:
    driver.get("https://www.megabox.co.kr/theater/list")

    if click_region_tab("제주"):
        theater_list_ul = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "li.on > div > ul")
        ))

        theater_links = theater_list_ul.find_elements(By.TAG_NAME, "a")
        hrefs = [link.get_attribute("href") for link in theater_links]

        for href in hrefs:
            print(href)
    else:
        print("해당 지역 탭을 찾을 수 없습니다.")

finally:
    driver.quit()

https://www.megabox.co.kr/theater?brchNo=0059
https://www.megabox.co.kr/theater?brchNo=0054
https://www.megabox.co.kr/theater?brchNo=0066


In [29]:
import requests
from bs4 import BeautifulSoup
import json
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

def get_cinema_info(brch_no):
    cinema = {}

    detail_url = f"https://www.megabox.co.kr/theater?brchNo={brch_no}"
    detail_res = requests.get(detail_url, headers=HEADERS)
    detail_res.raise_for_status()
    detail_soup = BeautifulSoup(detail_res.text, "html.parser")

    cinema_name_tag = detail_soup.select_one("#contents > div.theater-detail-page > div.theater-top > h2")
    address_tag = detail_soup.select_one("#contents > div.theater-detail-page > div.theater-all > p")

    cinema_name = cinema_name_tag.get_text(strip=True) if cinema_name_tag else ""
    address = address_tag.get_text(strip=True) if address_tag else ""

    cinema["cinema_name"] = cinema_name
    cinema["address"] = address

    time_url = f"https://www.megabox.co.kr/theater/time?brchNo={brch_no}"
    time_res = requests.get(time_url, headers=HEADERS)
    time_res.raise_for_status()
    time_soup = BeautifulSoup(time_res.text, "html.parser")

    movies = []

    movie_blocks = time_soup.select(".movie-list > ul > li")
    for movie in movie_blocks:
        title_tag = movie.select_one(".movie-title")
        title = title_tag.get_text(strip=True) if title_tag else ""

        showtimes = []
        time_tags = movie.select(".time-list > ul > li > a")
        for time_tag in time_tags:
            time_text = time_tag.get_text(strip=True)
            showtimes.append(time_text)

        movies.append({
            "title": title,
            "showtimes": showtimes
        })

    cinema["movies"] = movies

    return cinema


if __name__ == "__main__":
    brch_no_list = [
        "1372", "1341", "0090", "1431", "0041", "0073", "1572", "1581", "1311", "1211",
        "1331", "1371", "1381", "1202", "1561", "1321", "1351", "1212", "0083", "1562",
        "4121", "0029", "0053", "0035", "4152", "0039", "0019", "4451", "0089", "0084",
        "4104", "4722", "4221", "4631", "0086", "0051", "0052", "0042", "0062", "0060",
        "0036", "4291", "0088", "4253", "0020", "4821", "4462", "0070", "4112", "4132",
        "4115", "0064", "4651", "4041", "4062", "4001", "0080", "4051", "0085", "0027",
        "3141", "0018", "3021", "0028", "0009", "3011", "0017", "3391", "3392", "0008",
        "3631", "3651", "0056", "0049", "0069", "0068", "3501", "0040", "7122", "7303",
        "7401", "0074", "0067", "7011", "0022", "0072", "6161", "0076", "7451", "6001",
        "0061", "0025", "0032", "6642", "0082", "0063", "6262", "0057", "6191", "0045",
        "0048", "0014", "0038", "0079", "0087", "5021", "5061", "5302", "5401", "5552",
        "0010", "0050", "0021", "5064", "2001", "2171", "2202", "0037", "0059", "0054",
        "0066"
    ]

    results = []

    for brch_no in brch_no_list:
        print(f"크롤링 중: brchNo={brch_no}")
        try:
            info = get_cinema_info(brch_no)
            results.append(info)
        except Exception as e:
            print(f"에러 발생 (brchNo={brch_no}): {e}")

        time.sleep(0.5)

    # 학습용 JSON 형식으로 저장
    with open("megabox_cinemas_dataset.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("데이터 크롤링 및 저장 완료: megabox_cinemas_dataset.json")

크롤링 중: brchNo=1372
크롤링 중: brchNo=1341
크롤링 중: brchNo=0090
크롤링 중: brchNo=1431
크롤링 중: brchNo=0041
크롤링 중: brchNo=0073
크롤링 중: brchNo=1572
크롤링 중: brchNo=1581
크롤링 중: brchNo=1311
크롤링 중: brchNo=1211
크롤링 중: brchNo=1331
크롤링 중: brchNo=1371
크롤링 중: brchNo=1381
크롤링 중: brchNo=1202
크롤링 중: brchNo=1561
크롤링 중: brchNo=1321
크롤링 중: brchNo=1351
크롤링 중: brchNo=1212
크롤링 중: brchNo=0083
크롤링 중: brchNo=1562
크롤링 중: brchNo=4121
크롤링 중: brchNo=0029
크롤링 중: brchNo=0053
크롤링 중: brchNo=0035
크롤링 중: brchNo=4152
크롤링 중: brchNo=0039
크롤링 중: brchNo=0019
크롤링 중: brchNo=4451
크롤링 중: brchNo=0089
크롤링 중: brchNo=0084
크롤링 중: brchNo=4104
크롤링 중: brchNo=4722
크롤링 중: brchNo=4221
크롤링 중: brchNo=4631
크롤링 중: brchNo=0086
크롤링 중: brchNo=0051
크롤링 중: brchNo=0052
크롤링 중: brchNo=0042
크롤링 중: brchNo=0062
크롤링 중: brchNo=0060
크롤링 중: brchNo=0036
크롤링 중: brchNo=4291
크롤링 중: brchNo=0088
크롤링 중: brchNo=4253
크롤링 중: brchNo=0020
크롤링 중: brchNo=4821
크롤링 중: brchNo=4462
크롤링 중: brchNo=0070
크롤링 중: brchNo=4112
크롤링 중: brchNo=4132
크롤링 중: brchNo=4115
크롤링 중: brchNo=0064
크롤링 중: brchN

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import time

def get_cinema_data(brch_no, driver, wait):
    cinema = {}

    # 1. 극장 상세페이지에서 이름, 주소 가져오기
    detail_url = f"https://www.megabox.co.kr/theater?brchNo={brch_no}"
    driver.get(detail_url)
    try:
        cinema_name = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "#contents > div.theater-detail-page > div.theater-top > h2")
        )).text.strip()
    except:
        cinema_name = ""

    try:
        address = driver.find_element(
            By.CSS_SELECTOR, "#contents > div.theater-detail-page > div.theater-all > p"
        ).text.strip()
    except:
        address = ""

    cinema["cinema_name"] = cinema_name
    cinema["address"] = address

    # 2. 상영시간표 페이지에서 영화별 상영시간 가져오기
    time_url = f"https://www.megabox.co.kr/theater/time?brchNo={brch_no}"
    driver.get(time_url)

    # 영화 블록 로딩 대기
    movie_blocks = wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "#tab02 div.reserve.theater-list-box > div")
    ))

    movies = []
    for block in movie_blocks:
        try:
            title_elem = block.find_element(By.CSS_SELECTOR, "div.theater-tit p:nth-child(2) a")
            title = title_elem.text.strip()
        except:
            continue

        try:
            time_elems = block.find_elements(By.CSS_SELECTOR, "div.theater-time-box p.time")
            times = [t.text.strip() for t in time_elems if t.text.strip()]
        except:
            times = []

        movies.append({
            "title": title,
            "showtimes": times
        })

    cinema["movies"] = movies

    return cinema

if __name__ == "__main__":
    brch_no_list = [
        "1372", "1341", "0090", "1431", "0041", "0073", "1572", "1581", "1311", "1211",
        "1331", "1371", "1381", "1202", "1561", "1321", "1351", "1212", "0083", "1562",
        "4121", "0029", "0053", "0035", "4152", "0039", "0019", "4451", "0089", "0084",
        "4104", "4722", "4221", "4631", "0086", "0051", "0052", "0042", "0062", "0060",
        "0036", "4291", "0088", "4253", "0020", "4821", "4462", "0070", "4112", "4132",
        "4115", "0064", "4651", "4041", "4062", "4001", "0080", "4051", "0085", "0027",
        "3141", "0018", "3021", "0028", "0009", "3011", "0017", "3391", "3392", "0008",
        "3631", "3651", "0056", "0049", "0069", "0068", "3501", "0040", "7122", "7303",
        "7401", "0074", "0067", "7011", "0022", "0072", "6161", "0076", "7451", "6001",
        "0061", "0025", "0032", "6642", "0082", "0063", "6262", "0057", "6191", "0045",
        "0048", "0014", "0038", "0079", "0087", "5021", "5061", "5302", "5401", "5552",
        "0010", "0050", "0021", "5064", "2001", "2171", "2202", "0037", "0059", "0054",
        "0066"
    ]

    options = Options()
    options.add_argument("--headless")  # 창 안 띄우기
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    results = []

    for brch_no in brch_no_list:
        print(f"크롤링 중: brchNo={brch_no}")
        try:
            data = get_cinema_data(brch_no, driver, wait)
            results.append(data)
        except Exception as e:
            print(f"에러 발생 brchNo={brch_no}: {e}")
        time.sleep(0.5)  # 너무 빠른 요청 방지

    driver.quit()

    # JSON 저장
    with open("megabox_cinema_with_showtimes.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("크롤링 완료, megabox_cinema_with_showtimes.json 저장됨")

크롤링 중: brchNo=1372
크롤링 중: brchNo=1341
크롤링 중: brchNo=0090
크롤링 중: brchNo=1431
크롤링 중: brchNo=0041
크롤링 중: brchNo=0073
크롤링 중: brchNo=1572
크롤링 중: brchNo=1581
크롤링 중: brchNo=1311
크롤링 중: brchNo=1211
크롤링 중: brchNo=1331
크롤링 중: brchNo=1371
크롤링 중: brchNo=1381
크롤링 중: brchNo=1202
크롤링 중: brchNo=1561
크롤링 중: brchNo=1321
크롤링 중: brchNo=1351
크롤링 중: brchNo=1212
크롤링 중: brchNo=0083
크롤링 중: brchNo=1562
크롤링 중: brchNo=4121
크롤링 중: brchNo=0029
크롤링 중: brchNo=0053
크롤링 중: brchNo=0035
크롤링 중: brchNo=4152
크롤링 중: brchNo=0039
크롤링 중: brchNo=0019
크롤링 중: brchNo=4451
크롤링 중: brchNo=0089
크롤링 중: brchNo=0084
크롤링 중: brchNo=4104
크롤링 중: brchNo=4722
크롤링 중: brchNo=4221
크롤링 중: brchNo=4631
크롤링 중: brchNo=0086
크롤링 중: brchNo=0051
크롤링 중: brchNo=0052
크롤링 중: brchNo=0042
크롤링 중: brchNo=0062
크롤링 중: brchNo=0060
크롤링 중: brchNo=0036
크롤링 중: brchNo=4291
크롤링 중: brchNo=0088
크롤링 중: brchNo=4253
크롤링 중: brchNo=0020
크롤링 중: brchNo=4821
크롤링 중: brchNo=4462
크롤링 중: brchNo=0070
크롤링 중: brchNo=4112
크롤링 중: brchNo=4132
크롤링 중: brchNo=4115
크롤링 중: brchNo=0064
크롤링 중: brchN